# Q1. AI Resume Screening Assistant (LangChain + RAG)
- **Model:** `gemini-3.5-flash-lite`
- **Pipeline:** `PyPDFLoader` -> `RecursiveCharacterTextSplitter` -> `GoogleGenerativeAIEmbeddings` -> `FAISS` -> `Retriever` -> `ChatPromptTemplate` -> `JsonOutputParser`

In [2]:
import os
import json
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from fpdf import FPDF

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Load single GOOGLE_API_KEY from .env
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", google_api_key=GOOGLE_API_KEY, temperature=0
)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY
)

C:\Users\singh\AppData\Local\Temp\ipykernel_23784\646573162.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### Step 1: Create 3 Sample PDF Resumes (`Resume_A.pdf`, `Resume_B.pdf`, `Resume_C.pdf`)

In [3]:
resumes_data = {
    "Resume_A.pdf": "Name: Aarav Sharma\nRole: Data Scientist\nSkills: Python, SQL, Machine Learning, Deep Learning, PyTorch, LangChain, RAG, FAISS, Pandas.\nExperience: 3 years building ML and GenAI RAG pipelines.",
    "Resume_B.pdf": "Name: Rohan Verma\nRole: Software Developer\nSkills: Python, SQL, JavaScript, HTML, CSS, Pandas, Git.\nExperience: 2 years building web apps and basic data scripts.",
    "Resume_C.pdf": "Name: Priya Nair\nRole: Business Analyst Fresher\nSkills: Excel, Tableau, Basic SQL, PowerPoint.\nExperience: Fresh graduate with academic analytics projects.",
}

for filename, text in resumes_data.items():
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(0, 10, txt=text)
    pdf.output(filename)

print("Created sample PDF resumes:", list(resumes_data.keys()))

Created sample PDF resumes: ['Resume_A.pdf', 'Resume_B.pdf', 'Resume_C.pdf']


C:\Users\singh\AppData\Local\Temp\ipykernel_23784\1272967362.py:11: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.multi_cell(0, 10, txt=text)


### Step 2: Define RAG Pipeline & Output Parser

In [4]:
class ResumeEvaluation(BaseModel):
    candidate_name: str = Field(description="Name of the candidate")
    match_score: int = Field(description="Match score between 0 and 100")
    matching_skills: List[str] = Field(description="Skills matching the JD")
    missing_skills: List[str] = Field(
        description="Skills required in JD but missing in resume"
    )
    candidate_summary: str = Field(description="Brief candidate summary")
    strengths: List[str] = Field(description="Candidate strengths")
    weaknesses: List[str] = Field(description="Candidate weaknesses")
    hiring_recommendation: str = Field(
        description="Hiring decision with short justification"
    )


parser = JsonOutputParser(pydantic_object=ResumeEvaluation)

prompt = ChatPromptTemplate.from_template(
    "You are an AI Resume Screening Assistant. Answer ONLY from the resume context below.\n\n"
    "Job Description:\n{jd}\n\n"
    "Resume Context:\n{context}\n\n"
    "{format_instructions}"
)


def evaluate_resume(pdf_path: str, jd: str) -> dict:
    docs = PyPDFLoader(pdf_path).load()
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=50
    ).split_documents(docs)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever()
    retrieved_docs = retriever.invoke(jd)
    context = "\n\n".join([d.page_content for d in retrieved_docs])

    chain = prompt | llm | parser
    return chain.invoke(
        {
            "jd": jd,
            "context": context,
            "format_instructions": parser.get_format_instructions(),
        }
    )

### Step 3: Run Example Test Cases

In [5]:
jd = "Role: Data Scientist. Required Skills: Python, SQL, Machine Learning, Deep Learning, PyTorch, LangChain, RAG, FAISS."

res_a = evaluate_resume("Resume_A.pdf", jd)
res_b = evaluate_resume("Resume_B.pdf", jd)
res_c = evaluate_resume("Resume_C.pdf", jd)

print("1. Evaluate Resume A for Data Scientist Role:")
print(json.dumps(res_a, indent=2))

print("\n2. Compare Resume A and Resume B for the same JD:")
print(
    f"Resume A ({res_a['candidate_name']}): Score {res_a['match_score']}/100 | {res_a['hiring_recommendation']}"
)
print(
    f"Resume B ({res_b['candidate_name']}): Score {res_b['match_score']}/100 | {res_b['hiring_recommendation']}"
)

print("\n3. Identify Missing Skills in Resume C:")
print(res_c["missing_skills"])

print("\n4. Recommend the Best Candidate with Justification:")
best = max([res_a, res_b, res_c], key=lambda x: x["match_score"])
print(
    f"Best Candidate: {best['candidate_name']} (Score: {best['match_score']}/100) -> {best['hiring_recommendation']}"
)

c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1. Evaluate Resume A for Data Scientist Role:
{
  "candidate_name": "Aarav Sharma",
  "match_score": 100,
  "matching_skills": [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "PyTorch",
    "LangChain",
    "RAG",
    "FAISS"
  ],
  "missing_skills": [],
  "candidate_summary": "Data Scientist with 3 years of experience building ML and GenAI RAG pipelines, possessing all required skills.",
  "strengths": [
    "Possesses all required technical skills including Python, SQL, Machine Learning, Deep Learning, PyTorch, LangChain, RAG, and FAISS",
    "3 years of relevant experience building ML and GenAI RAG pipelines"
  ],
  "weaknesses": [
    "None identified based on the provided resume and job description"
  ],
  "hiring_recommendation": "Highly recommended for hire as the candidate matches 100% of the required skills for the Data Scientist role."
}

2. Compare Resume A and Resume B for the same JD:
Resume A (Aarav Sharma): Score 100/100 | Highly recommended f